# Agent Analytics Dashboard



##### Installing dependencies

In [ ]:
pip install streamlit

##### Webapp script

In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
from google.cloud import bigquery
import plotly.express as px
import plotly.graph_objects as go
import google.colab.auth

# --- 1. Page Configuration & Styling ---
st.set_page_config(page_title="Agent Analytics V2", layout="wide")
st.title("🤖 Agent Analytics & Orchestration Dashboard")

# --- 2. Sidebar: Global Configuration & Filters ---
with st.sidebar:
    st.header("Data Source")
    project_id = st.text_input("GCP Project ID", "REPLACE WITH YOUR PROJECT ID")
    dataset_id = st.text_input("Dataset ID", "REPLACE WITH YOUR DATASET ID")
    table_id = st.text_input("Table ID", "REPLACE WITH YOUR TABLE ID")
    table_ref = f"`{project_id}.{dataset_id}.{table_id}`"

    st.divider()
    st.header("Global Filters")
    time_hours = st.slider("Time Range (Hours)", 1, 720, 72)
    agent_name = st.text_input("Filter by Agent Name", "")

    # Base filter string
    where_base = f"timestamp >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL {time_hours} HOUR)"
    if agent_name:
        where_base += f" AND agent = '{agent_name}'"

# --- 3. Data Connector ---

# Authenticate user to Google Cloud
google.colab.auth.authenticate_user()

@st.cache_resource
def get_bq_client(p_id):
    return bigquery.Client(project=p_id)

client = get_bq_client(project_id)

def query(sql):
    return client.query(sql).to_dataframe()

# --- 4. Section: Executive Summary (Top-Level KPIs) ---
st.header("📊 Executive Summary")
kpi_data = query(f"""
    SELECT
        COUNT(DISTINCT session_id) as sessions,
        COUNT(DISTINCT trace_id) as traces,
        COUNT(DISTINCT invocation_id) as invocations,
        COUNTIF(event_type = 'LLM_REQUEST') as llm_calls,
        COUNTIF(event_type = 'TOOL_STARTING') as tool_calls,
        SAFE_DIVIDE(COUNTIF(status = 'ERROR' OR error_message IS NOT NULL), COUNT(*)) as error_rate
    FROM {table_ref} WHERE {where_base}
""")

c1, c2, c3, c4, c5, c6 = st.columns(6)
c1.metric("Sessions", kpi_data['sessions'][0])
c2.metric("Traces", kpi_data['traces'][0])
c3.metric("Invocations", kpi_data['invocations'][0])
c4.metric("LLM Calls", kpi_data['llm_calls'][0])
c5.metric("Tool Calls", kpi_data['tool_calls'][0])
c6.metric("Error Rate", f"{kpi_data['error_rate'][0]*100:.2f}%")

st.divider()

# --- 5. Section: Detailed Analytics Tabs ---
tab_usage, tab_perf, tab_error, tab_multi, tab_hitl, tab_trace = st.tabs([
    "📈 Usage & Volume", "⚡ Performance", "🛡️ Reliability", "🖼️ Multimodal", "🤝 HITL", "🔍 Trace Drill-down"
])

# --- TAB: USAGE & VOLUME ---
with tab_usage:
    col_u1, col_u2 = st.columns([2, 1])
    with col_u1:
        st.subheader("Token Consumption Trend")
        token_trend = query(f"""
            SELECT TIMESTAMP_TRUNC(timestamp, HOUR) as hour,
                   SUM(CAST(JSON_VALUE(content, '$.usage.prompt') AS INT64)) as prompt,
                   SUM(CAST(JSON_VALUE(content, '$.usage.completion') AS INT64)) as completion
            FROM {table_ref} WHERE event_type = 'LLM_RESPONSE' AND {where_base}
            GROUP BY 1 ORDER BY 1
        """)
        fig = px.area(token_trend, x="hour", y=["prompt", "completion"], title="Tokens over Time")
        st.plotly_chart(fig, width='stretch')

    with col_u2:
        st.subheader("Top Users")
        top_users = query(f"SELECT user_id, COUNT(*) as count FROM {table_ref} WHERE {where_base} GROUP BY 1 ORDER BY 2 DESC LIMIT 10")
        st.dataframe(top_users, width='stretch')

# --- TAB: PERFORMANCE ---
with tab_perf:
    st.subheader("Latency Percentiles (ms)")
    lat_data = query(f"""
        WITH enriched_events AS (
            SELECT
                *,
                CASE
                    WHEN event_type = 'USER_MESSAGE_RECEIVED' THEN 'user'
                    WHEN event_type IN ('LLM_REQUEST','LLM_RESPONSE','LLM_ERROR') THEN 'llm'
                    WHEN event_type IN ('TOOL_STARTING','TOOL_COMPLETED','TOOL_ERROR') THEN 'tool'
                    ELSE 'other'
                END AS event_family
            FROM {table_ref}
            WHERE {where_base}
        )
        SELECT
            APPROX_QUANTILES(CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), 100)[OFFSET(50)] as p50,
            APPROX_QUANTILES(CAST(JSON_VALUE(latency_ms, '$.total_ms') AS FLOAT64), 100)[OFFSET(90)] as p90,
            AVG(CAST(JSON_VALUE(latency_ms, '$.time_to_first_token_ms') AS FLOAT64)) as avg_ttft
        FROM enriched_events WHERE event_family = 'llm'
    """)
    p1, p2, p3 = st.columns(3)
    p1.metric("p50 Latency", f"{lat_data['p50'][0]:.0f}ms")
    p2.metric("p90 Latency", f"{lat_data['p90'][0]:.0f}ms")
    p3.metric("Avg TTFT", f"{lat_data['avg_ttft'][0]:.0f}ms")

# --- TAB: RELIABILITY ---
with tab_error:
    st.subheader("Errors by Model & Tool")
    err_data = query(f"""
        SELECT JSON_VALUE(attributes, '$.model') as model, COUNT(*) as errors
        FROM {table_ref} WHERE (status = 'ERROR' OR error_message IS NOT NULL) AND {where_base}
        GROUP BY 1 ORDER BY 2 DESC
    """)
    st.plotly_chart(px.pie(err_data, values='errors', names='model', title="Error Distribution by Model"))

# --- TAB: MULTIMODAL ---
with tab_multi:
    st.subheader("Multimodal Content Analysis")
    multi_data = query(f"""
        SELECT cp.mime_type, COUNT(*) as count
        FROM {table_ref}, UNNEST(content_parts) as cp
        WHERE {where_base} GROUP BY 1
    """)
    st.plotly_chart(px.bar(multi_data, x='mime_type', y='count', title="MIME Type Breakdown"))

# --- TAB: HITL ---
with tab_hitl:
    st.subheader("Human-in-the-Loop Orchestration")
    hitl_data = query(f"""
        WITH enriched_events AS (
            SELECT
                *,
                CASE
                    WHEN event_type = 'USER_MESSAGE_RECEIVED' THEN 'user'
                    WHEN event_type IN ('LLM_REQUEST','LLM_RESPONSE','LLM_ERROR') THEN 'llm'
                    WHEN event_type IN ('TOOL_STARTING','TOOL_COMPLETED','TOOL_ERROR') THEN 'tool'
                    ELSE 'other'
                END AS event_family
            FROM {table_ref}
            WHERE {where_base}
        )
        SELECT event_type, COUNT(*) as count
        FROM enriched_events WHERE event_family = 'other' # Assuming 'hitl' events fall under 'other' category as per Flask app logic.
        GROUP BY 1
    """)
    st.table(hitl_data)

# --- TAB: TRACE DRILL-DOWN ---
with tab_trace:
    st.subheader("Session Explorer")
    sessions = query(f"SELECT session_id, agent, MIN(timestamp) as start_ts FROM {table_ref} WHERE {where_base} GROUP BY 1, 2 ORDER BY 3 DESC LIMIT 50")
    selected_sid = st.selectbox("Select Session ID", sessions['session_id'])

    if selected_sid:
        trace_detail = query(f"SELECT timestamp, event_type, agent, status, error_message FROM {table_ref} WHERE session_id = '{selected_sid}' ORDER BY timestamp")
        st.dataframe(trace_detail)


Overwriting app.py


##### Run app.py

In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧your url is: https://tame-guests-fetch.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.237.66.184:8501

y
